# Bruecken-Tabellen-Analyse

Misst, wie oft in Spider-Train/-Val/-Ent eine Gold-Tabelle nur ueber eine JOIN-ON-Bedingung
ins Schema kommt, aber keine Spalte zu SELECT/WHERE/GROUP BY/ORDER BY/HAVING beisteuert.
Fuer einen rein spalten-relevanz-klassifizierenden Ansatz wie ExSL hat eine solche Tabelle
kein lexikalisches Signal aus der Frage - sie ist ein Kandidat fuer die Graph-Pfadsuche
(SchemaGraphSQL-Idee), die nach der Klassifikation fehlende Bruecken-Tabellen ueber den
FK-Graphen ergaenzen soll.


In [2]:
import re
from collections import defaultdict

import pandas as pd

from src.utils import get_gold_schema, _extract_tables, _collect_refs
from src.data_loaders.spider_data import spider_train, spider_val, spider_tables, get_spider_schema_ddl_and_candidates
from src.data_loaders.spider_ent_data import spider_ent as spider_ent_raw

spider_schema_ddls_and_candidates = get_spider_schema_ddl_and_candidates()

print(f"Spider-Train: {len(spider_train)} Fragen")
print(f"Spider-Val:   {len(spider_val)} Fragen")
print(f"Spider-Ent:   {len(spider_ent_raw)} Fragen")


D:\Master\Grundprojekt\Grundprojekt_Schema_Linking_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Spider-Train: 7000 Fragen
Spider-Val:   1034 Fragen
Spider-Ent:   602 Fragen


## Non-ON-Parser

Wie `utils.parse_orig_sql`, aber ignoriert Spaltenreferenzen, die nur in ON-Klauseln
(Join-Bedingungen) auftauchen. Nutzt bewusst dieselben privaten Hilfsfunktionen aus
`utils.py` (`_extract_tables`, `_collect_refs`) fuer Tabellen-/Alias-Aufloesung, damit
das Ergebnis konsistent zur Produktionslogik bleibt - der einzige Unterschied ist, dass
`_collect_on_refs` (die ON-Klausel-Sammlung) hier nicht aufgerufen wird.

Einschraenkung: Subqueries innerhalb von FROM/SELECT/WHERE greifen intern wieder auf
`utils._walk_query` (die volle, ON-einschliessende Logik) zurueck. Das ist in Spider selten
und wird hier bewusst in Kauf genommen statt den kompletten Parser zu duplizieren.


In [2]:
def _handle_select_no_on(node, result):
    alias_map = {}
    tables = []
    _extract_tables(node.get('from'), alias_map, tables, result)

    refs = []
    for key in ('select', 'select_distinct', 'where', 'groupby', 'orderby', 'having'):
        _collect_refs(node.get(key), refs, result)

    for ref in refs:
        if '.' in ref:
            alias, col = ref.split('.', 1)
            tbl = alias_map.get(alias, alias).lower()
            result[tbl].add(col.lower())
        elif len(tables) == 1:
            result[tables[0].lower()].add(ref.lower())
    # Bewusst KEIN Fallback-Eintrag fuer Tabellen ohne Non-ON-Spalte (anders als
    # utils._handle_select) - das Fehlen eines Eintrags ist hier genau das gesuchte Signal.


def _walk_query_no_on(node, result):
    if not isinstance(node, dict):
        return
    for op in ('union', 'union_all', 'intersect', 'except'):
        if op in node:
            items = node[op] if isinstance(node[op], list) else [node[op]]
            for item in items:
                _walk_query_no_on(item, result)
            return
    if 'select' in node or 'select_distinct' in node:
        _handle_select_no_on(node, result)


def parse_non_on_refs(sql) -> dict:
    from mo_sql_parsing import parse as mo_parse
    try:
        tree = mo_parse(sql)
    except Exception:
        result = defaultdict(set)
        for part in re.split(r'\b(UNION ALL|UNION|INTERSECT|EXCEPT)\b', sql, flags=re.IGNORECASE):
            part = part.strip()
            if not part or part.upper() in ('UNION', 'UNION ALL', 'INTERSECT', 'EXCEPT'):
                continue
            try:
                sub_tree = mo_parse(part)
                _walk_query_no_on(sub_tree, result)
            except Exception:
                continue
        return dict(result)
    result = defaultdict(set)
    _walk_query_no_on(tree, result)
    return dict(result)


def find_bridge_tables(sql: str, gold_schema: dict) -> list:
    """Gold-Tabellen, die keine Spalte ausserhalb von JOIN-ON-Bedingungen beisteuern."""
    if len(gold_schema) < 2:
        return []
    non_on = parse_non_on_refs(sql)
    non_on = {t.lower(): cols for t, cols in non_on.items()}
    return [t for t in gold_schema if not non_on.get(t.lower())]


## FK-Bestaetigung

Zusaetzlicher Validitaets-Check: hat die erkannte Bruecken-Tabelle tatsaechlich eine
direkte FK-Kante zu einer anderen Gold-Tabelle? Falls nicht, ist der Fund eher ein
Parsing-Artefakt (z.B. Join ueber eine nicht als FK hinterlegte Spalte) als eine echte
strukturelle Bruecke - dient hier nur der manuellen Pruefung der Definition.


In [3]:
def build_fk_adjacency(db_id: str) -> dict:
    db = next((d for d in spider_tables if d['db_id'] == db_id), None)
    if db is None:
        return {}
    orig_tables = db['table_names_original']
    col_names = db['column_names_original']
    adjacency = defaultdict(set)
    for fk_from, fk_to in db['foreign_keys']:
        t_from = orig_tables[col_names[fk_from][0]].lower()
        t_to = orig_tables[col_names[fk_to][0]].lower()
        adjacency[t_from].add(t_to)
        adjacency[t_to].add(t_from)
    return adjacency


def bridge_confirmed_by_fk(db_id: str, bridge_table: str, gold_tables) -> bool:
    adjacency = build_fk_adjacency(db_id)
    neighbors = adjacency.get(bridge_table.lower(), set())
    others = {t.lower() for t in gold_tables if t.lower() != bridge_table.lower()}
    return bool(neighbors & others)


## Analyse ueber alle drei Datensaetze

In [4]:
def analyze(dataset, sql_key, db_key, label, extra_field_fn=None):
    rows = []
    for q in dataset:
        sql = q[sql_key]
        db_id = q[db_key]
        db_tables = spider_schema_ddls_and_candidates.get(db_id)
        if db_tables is None:
            continue
        gold_schema = get_gold_schema(sql, db_tables)
        bridges = find_bridge_tables(sql, gold_schema)
        confirmed = [b for b in bridges if bridge_confirmed_by_fk(db_id, b, gold_schema.keys())]
        row = {
            "dataset": label,
            "db_id": db_id,
            "sql": sql,
            "num_gold_tables": len(gold_schema),
            "bridge_tables": bridges,
            "num_bridge_tables": len(bridges),
            "bridge_tables_fk_confirmed": confirmed,
            "has_bridge": len(bridges) > 0,
        }
        if extra_field_fn is not None:
            row.update(extra_field_fn(q))
        rows.append(row)
    return pd.DataFrame(rows)


df_train = analyze(spider_train, 'query', 'db_id', 'spider_train')
df_val = analyze(spider_val, 'query', 'db_id', 'spider_val')
df_ent = analyze(
    spider_ent_raw, 'original_SQL', 'eval_db', 'spider_ent',
    extra_field_fn=lambda q: {"data_asset": q['data_asset']},
)

df_all = pd.concat([df_train, df_val, df_ent], ignore_index=True)
print(f"ausgewertet: {len(df_all)} / {len(spider_train) + len(spider_val) + len(spider_ent_raw)} Fragen "
      f"({len(spider_train) + len(spider_val) + len(spider_ent_raw) - len(df_all)} uebersprungen, db_id nicht in tables.json gefunden)")


ausgewertet: 8636 / 8636 Fragen (0 uebersprungen, db_id nicht in tables.json gefunden)


## Zusammenfassung pro Datensatz

In [5]:
summary = df_all.groupby('dataset').agg(
    n_queries=('sql', 'count'),
    n_multi_table=('num_gold_tables', lambda s: (s > 1).sum()),
    n_with_bridge=('has_bridge', 'sum'),
).reindex(['spider_train', 'spider_val', 'spider_ent']).reset_index()

summary['pct_of_multi_table_queries'] = (summary['n_with_bridge'] / summary['n_multi_table'] * 100).round(1)
summary['pct_of_all_queries'] = (summary['n_with_bridge'] / summary['n_queries'] * 100).round(1)
summary


,dataset,n_queries,n_multi_table,n_with_bridge,pct_of_multi_table_queries,pct_of_all_queries
0,spider_train,7000,3069,1290,42.0,18.4
1,spider_val,1034,459,202,44.0,19.5
2,spider_ent,602,241,95,39.4,15.8


In [6]:
bridge_rows = df_all[df_all['num_bridge_tables'] > 0].copy()
bridge_rows['num_bridge_confirmed'] = bridge_rows['bridge_tables_fk_confirmed'].apply(len)

n_found = bridge_rows['num_bridge_tables'].sum()
n_confirmed = bridge_rows['num_bridge_confirmed'].sum()
print(f"FK-bestaetigt: {n_confirmed} / {n_found} erkannte Bruecken-Tabellen-Vorkommen "
      f"({n_confirmed / n_found * 100:.1f}%)")

print()
print("Nicht FK-bestaetigte Faelle nach Datensatz (moegliche Parsing-Artefakte, zur manuellen Pruefung):")
not_confirmed = bridge_rows[bridge_rows['num_bridge_tables'] > bridge_rows['num_bridge_confirmed']]
not_confirmed['dataset'].value_counts()


FK-bestaetigt: 1743 / 1811 erkannte Bruecken-Tabellen-Vorkommen (96.2%)

Nicht FK-bestaetigte Faelle nach Datensatz (moegliche Parsing-Artefakte, zur manuellen Pruefung):


dataset
spider_train    57
spider_val      10
spider_ent       1
Name: count, dtype: int64

## Spider-Ent nach Domain

In [7]:
ent_by_domain = df_ent.groupby('data_asset').agg(
    n_queries=('sql', 'count'),
    n_multi_table=('num_gold_tables', lambda s: (s > 1).sum()),
    n_with_bridge=('has_bridge', 'sum'),
).reset_index()
ent_by_domain['pct_of_multi_table_queries'] = (ent_by_domain['n_with_bridge'] / ent_by_domain['n_multi_table'] * 100).round(1)
ent_by_domain.sort_values('pct_of_multi_table_queries', ascending=False)


,data_asset,n_queries,n_multi_table,n_with_bridge,pct_of_multi_table_queries
6,government_and_public_affairs,10,4,3,75.0
11,transportation_and_logistics,74,35,26,74.3
4,education_and_campus_management,42,10,7,70.0
0,animals_and_pets,70,49,27,55.1
3,document_and_content_management,58,17,8,47.1
7,military_warfare_losses,13,7,3,42.9
9,social_networks_and_reviews,45,22,6,27.3
5,geography_environment_and_climate,77,16,3,18.8
1,arts_culture_and_media,151,56,10,17.9
10,sports_and_athletics,54,22,2,9.1


## Beispiele zur manuellen Pruefung

In [8]:
pd.set_option('display.max_colwidth', None)

examples = df_all[df_all['has_bridge']].sample(min(8, df_all['has_bridge'].sum()), random_state=0)
for _, row in examples.iterrows():
    print(f"[{row['dataset']}] db={row['db_id']}")
    print(row['sql'])
    print(f"Bruecken-Tabellen: {row['bridge_tables']}  (FK-bestaetigt: {row['bridge_tables_fk_confirmed']})")
    print()


[spider_train] db=music_2
SELECT T2.firstname ,  T2.lastname FROM Performance AS T1 JOIN Band AS T2 ON T1.bandmate  =  T2.id JOIN Songs AS T3 ON T3.SongId  =  T1.SongId WHERE T3.Title  =  "Badlands"
Bruecken-Tabellen: ['performance']  (FK-bestaetigt: ['performance'])

[spider_ent] db=pets_1
select count(*) ,  t1.stuid from student as t1 join has_pet as t2 on t1.stuid  =  t2.stuid group by t1.stuid
Bruecken-Tabellen: ['has_pet']  (FK-bestaetigt: ['has_pet'])

[spider_train] db=college_1
SELECT DISTINCT T1.EMP_FNAME ,  T1.EMP_DOB FROM employee AS T1 JOIN CLASS AS T2 ON T1.EMP_NUM  =  T2.PROF_NUM WHERE CRS_CODE  =  "ACCT-211"
Bruecken-Tabellen: ['class']  (FK-bestaetigt: ['class'])

[spider_val] db=flight_2
SELECT T1.Airline FROM AIRLINES AS T1 JOIN FLIGHTS AS T2 ON T1.uid  =  T2.Airline GROUP BY T1.Airline ORDER BY count(*) DESC LIMIT 1
Bruecken-Tabellen: ['flights']  (FK-bestaetigt: [])

[spider_train] db=machine_repair
SELECT T3.Name ,  T2.Machine_series FROM repair_assignment AS T1 JO

## Testen der Logik ohne Gold-SQL für Inference

In [3]:
from src.utils import add_missing_bridge_tables



db_tables = spider_schema_ddls_and_candidates['machine_repair']
add_missing_bridge_tables({'technician': ['name', 'technician_ID'], 'machine': ['machine_series', 'machine_id']}, db_tables)

{'technician': ['name', 'technician_ID'],
 'machine': ['machine_series', 'machine_id'],
 'repair_assignment': ['machine_id', 'technician_id']}